<a href="https://colab.research.google.com/github/louis011209-1901/python_KO-01/blob/main/Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip list

In [ ]:
import pandas as pd
import geopandas as gpd

In [ ]:
pfad = r'/content/data/messpunkte.csv'
df = pd.read_csv(pfad)

In [ ]:
df

,name,ost,nord,hoehe
0,Zimmerwald,2602025,1191761,898
1,Bantiger,2606806,1202974,947
2,Gurten,2600378,1196252,858
3,Chasseral,2571227,1220302,1606
4,Napf,2638131,1205962,1406


In [ ]:
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['ost'], df['nord']),
    crs="EPSG:2056"
)

In [ ]:
print(gdf.crs)

EPSG:2056


In [ ]:
gdf.crs

<Projected CRS: EPSG:2056>
Name: CH1903+ / LV95
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Liechtenstein; Switzerland.
- bounds: (5.95, 45.81, 10.5, 47.81)
Coordinate Operation:
- name: Swiss Oblique Mercator 1995
- method: Hotine Oblique Mercator (variant B)
Datum: CH1903+
- Ellipsoid: Bessel 1841
- Prime Meridian: Greenwich

In [ ]:
gdf

,name,ost,nord,hoehe,geometry
0,Zimmerwald,2602025,1191761,898,POINT (2602025 1191761)
1,Bantiger,2606806,1202974,947,POINT (2606806 1202974)
2,Gurten,2600378,1196252,858,POINT (2600378 1196252)
3,Chasseral,2571227,1220302,1606,POINT (2571227 1220302)
4,Napf,2638131,1205962,1406,POINT (2638131 1205962)


In [ ]:
gdf_wgs84 = gdf.to_crs("EPSG:4326")

In [ ]:
print(gdf_wgs84.geometry[0])

POINT (7.465197704473807 46.87696822762355)


In [ ]:
gdf_wgs84

,name,ost,nord,hoehe,geometry
0,Zimmerwald,2602025,1191761,898,POINT (7.4652 46.87697)
1,Bantiger,2606806,1202974,947,POINT (7.52808 46.9778)
2,Gurten,2600378,1196252,858,POINT (7.4436 46.91737)
3,Chasseral,2571227,1220302,1606,POINT (7.05938 47.13307)
4,Napf,2638131,1205962,1406,POINT (7.94002 47.00362)


In [ ]:
from shapely.geometry import Point

In [ ]:
p = Point(2600000, 1200000)

In [ ]:
print(p.x, p.y)

2600000.0 1200000.0


In [ ]:
p2 = Point(2601500, 1201200)

In [ ]:
print(f"Distanz: {p.distance(p2):.1f} m")

Distanz: 1920.9 m


In [ ]:
buffer = p.buffer(5000)

In [ ]:
print(f"Pufferfläche: {buffer.area / 1e6:.2f} km²")

Pufferfläche: 78.41 km²


In [ ]:
pfad = r'/content/data/messgebiete.geojson'
zonen = gpd.read_file(pfad)

In [ ]:
result = gpd.sjoin(gdf, zonen, how="left", predicate="within")

In [ ]:
print(result[["name_left", "name_right"]])

    name_left name_right
0  Zimmerwald   Zone Süd
1    Bantiger  Zone Nord
2      Gurten  Zone Nord
3   Chasseral        NaN
4        Napf   Zone Ost


In [ ]:
import folium

In [ ]:
mitte = gdf_wgs84.geometry.union_all().centroid

In [ ]:
karte = folium.Map(
    location=[mitte.y, mitte.x],
    zoom_start=10,
    tiles="https://wmts.geo.admin.ch/1.0.0/ch.swisstopo.pixelkarte-farbe/default/current/3857/{z}/{x}/{y}.jpeg",
    attr="swisstopo",
 )

In [ ]:
for _, row in gdf_wgs84.iterrows():
  folium.Marker(
      location=[row.geometry.y, row.geometry.x],
      tooltip=f"{row['name']}: {row['hoehe']} M.ü.M."
  ).add_to(karte)

In [ ]:
karte.save("messpunkte.html")

In [ ]:
gdf.to_file("messpunkte.geojson", driver="GeoJSON")

In [ ]:
gdf.to_file("messpunkte.gpkg", driver="GPKG")